# 02 · Confusion-matrix & error analysis

**Person C.** Load a trained checkpoint, evaluate on the leakage-safe val split and on the webcam test set, and dig into the known-confusable groups (**M/N/S/T, A/E, K/V**) and the train→webcam gap.

Run from the repo root.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, torch
from torch.utils.data import DataLoader
from src.data import CLASSES, ASLImageDataset, frame_range_split
from src.evaluate import (load_checkpoint, predict_all, confusion_matrix,
                          per_class_accuracy, macro_f1, report, _webcam_samples)

CKPT = '../checkpoints/efficientnet_b0.pt'
ROOT = '../data/raw/asl_alphabet_train'
WEBCAM = '../data/webcam_testset'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_checkpoint(CKPT, device)

## Leakage-safe val split (dev only — not the reported number)

In [ ]:
m = frame_range_split(ROOT, 0.8)
val = ASLImageDataset(m.val, train=False)
yt, yp = predict_all(model, DataLoader(val, batch_size=64), device)
cm_val = confusion_matrix(yt, yp, len(CLASSES))
report(cm_val)

## Webcam test set — THE reported benchmark (target ≥ 90%)

In [ ]:
wc = ASLImageDataset(_webcam_samples(WEBCAM), train=False)
yt2, yp2 = predict_all(model, DataLoader(wc, batch_size=64), device)
cm_wc = confusion_matrix(yt2, yp2, len(CLASSES))
report(cm_wc)

## Where does the model confuse M/N/S/T, A/E, K/V?

In [ ]:
import matplotlib.pyplot as plt
for name, cm in [('val', cm_val), ('webcam', cm_wc)]:
    plt.figure(figsize=(10,9)); plt.imshow(cm, cmap='viridis')
    plt.xticks(range(29), CLASSES, rotation=90); plt.yticks(range(29), CLASSES)
    plt.title(f'confusion — {name}'); plt.xlabel('pred'); plt.ylabel('true')
    plt.colorbar(); plt.tight_layout(); plt.show()

## TODO
- Quantify the val→webcam accuracy gap and hypothesise causes.
- Pull the worst-confused image pairs and eyeball them.
- Feed findings back to Person A (augmentation) and Person B (training).